In [0]:
-- Outlet dimension: one row per outlet
CREATE OR REPLACE TABLE workspace.default.silver_dim_outlet AS
SELECT
  OutletID                           AS outlet_id,
  MAX(EstablishmentYear)             AS established_year,
  COALESCE(MAX(OutletSize),'Unknown') AS outlet_size,
  MAX(LocationType)                  AS location_tier,
  MAX(OutletType)                    AS outlet_type
FROM workspace.default.train_set
GROUP BY OutletID;

-- Fact table: one row per product per outlet
CREATE OR REPLACE TABLE workspace.default.silver_fact_sales AS
SELECT
  ProductID                          AS product_id,
  OutletID                           AS outlet_id,
  NULLIF(ProductVisibility, 0)       AS visibility,
  MRP                                AS price,
  OutletSales                        AS sales
FROM workspace.default.train_set;



SELECT
  (SELECT COUNT(*) FROM workspace.default.silver_dim_outlet) AS outlets,
  (SELECT COUNT(*) FROM workspace.default.silver_fact_sales) AS fact_rows,
  (SELECT COUNT_IF(visibility IS NULL) FROM workspace.default.silver_fact_sales) AS null_visibility,
  (SELECT COUNT(*) FROM workspace.default.silver_fact_sales f
     LEFT JOIN workspace.default.silver_dim_product p USING (product_id)
     WHERE p.product_id IS NULL) AS orphan_products;